In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="7"

# Flow

We input images and masks. The mask and image size must match, but between images they can have different sizes.

> We want to know which patches correspond to the mask area. So do we resize both the image and the mask and overlap the patches?
> Answer: Yes. This is what we do in the original. So might as well during preprocessing already compute the resized mask and return tensors that have the mask as well as original shape bounding box information. These will be sized for the entire resized image and so will be larger than required.

Compute the valid size for the image, resize and pad the image and mask in parallel. Batch-wise compute mask overlaps using the area interpolate method for all segmentations at once.

Batch-wise compute the patch bounding boxes in the original image sizes. We can first compute the patch bounding boxes for the resized images and then scale based on the scale parameters from the resize.

Finally, for each we need to create the valid mask.

All of these should be 1D arrays that correspond to the shape of the output from DINOv3.

In [3]:
import torch

# Preprocessing

## Preprocess Images

In [4]:
from aidan_lib.models.dino_lib_compiled import preprocess_imgs, get_compiled_dino, get_onnx_dino, get_normal_dino

Unable to import quantization op. Please install modelopt library (https://github.com/NVIDIA/TensorRT-Model-Optimizer?tab=readme-ov-file#installation) to add support for compiling quantized models


In [5]:
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"PyTorch CUDA Version: {torch.version.cuda}")
print(f"Device Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

CUDA Available: True
PyTorch CUDA Version: 12.4
Device Name: NVIDIA H100 80GB HBM3


In [6]:
# dino = get_compiled_dino("facebook/dinov3-vits16-pretrain-lvd1689m", "cuda", max_side_len=256)

In [7]:
batch_size=64
input_size = 1024

In [8]:
dino = get_onnx_dino(device="cuda", max_side_len=input_size, warmup_batch_size=batch_size)

Loading ONNX model from onnx-community/dinov3-vits16-pretrain-lvd1689m-ONNX...
Creating ONNX Runtime Inference Session...


2026-05-13 16:33:03.819587400 [W:onnxruntime:, transformer_memcpy.cc:111 ApplyImpl] 98 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.


Performing warmup pass...
ONNX loading and warmup complete.


In [9]:
dummy_input = torch.randn(batch_size, 3, input_size, input_size, dtype=torch.float32, device="cuda")

In [10]:
out = dino(dummy_input)

In [11]:
print(out[0].shape, out[1].shape)

torch.Size([64, 4101, 384]) torch.Size([64, 384])


In [12]:
normal_dino = get_normal_dino(device="cuda")

Loading facebook/dinov3-vits16-pretrain-lvd1689m...


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 11673.28it/s]


In [13]:
import time
start = time.perf_counter()
out = normal_dino(dummy_input)
print(time.perf_counter() - start)

OutOfMemoryError: CUDA out of memory. Tried to allocate 386.00 MiB. GPU 0 has a total capacity of 79.10 GiB of which 302.00 MiB is free. Including non-PyTorch memory, this process has 78.79 GiB memory in use. Of the allocated memory 7.64 GiB is allocated by PyTorch, and 231.29 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
out.last_hidden_state.shape